> ## SOLUTION / ANSWER KEY &mdash; Lab 8.10
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-10-observability.ipynb`](../lab-10-observability.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 8.10 &mdash; Observability for a Team

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 4 &middot; Module 8 &mdash; Multi-Agent Collaboration &amp; Decision Making**

### What you'll do
- Log every agent action into an auditable trace
- Count calls per agent to find the busy/faulty one
- Read a REAL agent trace and detect a runaway handoff loop

> **How this lab works (near-real):** read the **Concept**, fill the real `___` blanks in **Build it**, then **run it and read what happened**. The multi-agent *mechanics* (routing, shared state, voting, critique, synthesis, observability) are **real Python you build and run**; the **specialist agents and the assembled chatbot are real `create_agent` agents** that really answer. Finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is a working team and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langgraph`) and, for the specialists, a **real hosted model** &mdash; `ChatGroq(model="openai/gpt-oss-20b")` with your `GROQ_API_KEY` from `.env`. If the key is missing, the live cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. You are building the **multi-agent customer-service chatbot** &mdash; the client's Lab 4.2.

**Reference:** [Module 8 slides &mdash; Failure modes & observability](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 8 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (Day-4 provider)

WORK = "/tmp/biaa-lab-08-10"
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is present. The live specialist cells self-skip when it is absent."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-4 provider: a REAL hosted model with reliable tool-calling via create_agent.
# gpt-oss-20b is verified; do NOT use llama-3.3-70b-versatile (it 400s through create_agent).
# One shared model is fine -- each specialist differs by its system_prompt + its own tools.
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0) if groq_ready() else None

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("GROQ_API_KEY loaded | model: openai/gpt-oss-20b | WORK:", WORK)
else:
    print("GROQ_API_KEY NOT set -- add it to .env (free at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
A multi-agent system fails in new ways &mdash; **handoff loops**, agents **talking past** each other,
**cost blow-ups**, and **lost accountability** (deck slide 19). The defence is **observability**: log every
agent, message, handoff and decision so you can **replay** the conversation, find the **faulty agent**, and
watch **cost**. **LangSmith / Langfuse** do this for real graphs; here you build the offline version, then
point it at a **real** agent trace.

In [ ]:
from collections import Counter
print("we log (agent, action, detail) events and watch for loops")

## Build it
Complete the `AgentTrace` (log + read back) and `detect_loop` (a runaway handoff).

In [ ]:
from collections import Counter

class AgentTrace:
    def __init__(self):
        self.events = []
    def log(self, agent, action, detail):
        self.events.append((agent, action, detail))
    def agents_involved(self):
        return [a for a, _, _ in self.events]
    def calls_per_agent(self):
        return Counter(a for a, _, _ in self.events)

def detect_loop(path, limit=2):
    # a runaway loop: some agent appears MORE than `limit` times (a normal 2x back-and-forth is fine)
    return any(c > limit for c in Counter(path).values())

try:
    tr = AgentTrace()
    tr.log("supervisor", "route", "billing+tech")
    tr.log("billing", "tool", "lookup_invoice")
    tr.log("tech", "tool", "known_issues")
    print("involved :", tr.agents_involved())
    print("calls    :", dict(tr.calls_per_agent()))
    print("loop?    :", detect_loop(["a", "b", "a", "b", "a", "b"]))
    print("healthy? :", detect_loop(["supervisor", "billing", "tech"]))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Now feed your `AgentTrace` a **real** `create_agent` run: build one specialist, invoke it, and log every tool call it actually made. Your loop-detector reads a real path.

_This calls the real `openai/gpt-oss-20b` via Groq. If `GROQ_API_KEY` is unset the cell prints how to set it instead of crashing. Multi-agent runs make several model calls &mdash; keep live runs light on the free tier._

In [ ]:
if not groq_ready():
    print("GROQ_API_KEY not set -- add it to .env (free at console.groq.com), then re-run this cell.")
else:
    try:
        # The SAME idea, but reading a REAL create_agent trace: build one specialist, run it,
        # and log every tool call the model actually made. (Lab 8.1's specialist, observed.)
        from langchain_core.tools import tool
        from langchain.agents import create_agent
        INVOICES = {"4471": [{"amount": 50, "date": "Jul 01"}, {"amount": 50, "date": "Jul 01"}]}

        @tool
        def lookup_invoice(order_id: str) -> str:
            """Look up the charges on an order by its id. Use for billing / charge questions."""
            return str(INVOICES.get(order_id.strip(), [])) or "no charges"

        billing_agent = create_agent(llm, [lookup_invoice],
            system_prompt="You are the billing specialist. Use ONLY your own tools; answer in one sentence.")
        result = billing_agent.invoke(
            {"messages": [("user", "Was I double-charged on order 4471?")]}, config={"recursion_limit": 8})

        tr = AgentTrace()
        for m in result["messages"]:
            for tc in (getattr(m, "tool_calls", None) or []):
                tr.log("billing", "tool_call", tc["name"])         # log the REAL tool calls from the trace
        print("real agent events :", tr.events)
        print("calls per agent   :", dict(tr.calls_per_agent()))
        print("runaway loop?     :", detect_loop(tr.agents_involved()))
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- `agents_involved` and `calls_per_agent` read straight off the log &mdash; your #1 way to spot the busy or faulty agent.
- `detect_loop` allows a normal 2x back-and-forth but flags anything hotter &mdash; a runaway team caught before it burns tokens.
- The **live** cell logged the *real* tool calls from a real trace &mdash; exactly what LangSmith / Langfuse capture, in miniature.

## Your turn (open task &mdash; no grader)
Extend `AgentTrace` with a `cost` field per event (e.g. a token estimate) and a `total_cost()` method, then log a
few events. **What good looks like:** you can now answer *"which agent cost the most?"* from the trace &mdash; the
accountability a multi-agent system needs. Feed `detect_loop` a hand-built looping path to confirm it fires.

In [ ]:
# --- Reference answer (ONE good way to do the 'Your turn' task -- compare with your own) ---
class CostTrace(AgentTrace):
    def log(self, agent, action, detail, cost=0):
        self.events.append((agent, action, detail, cost))     # add a per-event cost
    def agents_involved(self):
        return [e[0] for e in self.events]
    def total_cost(self):
        return sum(e[3] for e in self.events)

ct = CostTrace()
ct.log("supervisor", "route", "billing+tech", cost=5)
ct.log("billing", "tool", "lookup_invoice", cost=12)
ct.log("tech", "tool", "known_issues", cost=9)
print("total cost:", ct.total_cost(), "| involved:", ct.agents_involved())
print("loop?     :", detect_loop(["a", "b", "a", "b", "a"]))   # hand-built runaway -> True

---
### Key takeaway
Log every agent, message, handoff and decision so you can replay the conversation, find the faulty agent and watch cost. A multi-agent system is only as trustworthy as it is observable.

[&#8592; All Module 8 labs](./index.html) &nbsp;&middot;&nbsp; [Module 8 slides](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>